In [1]:
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors


In [2]:
train_df = pd.read_csv('train.csv')
df = train_df.copy()

In [3]:
import tiktoken
tokenizer = tiktoken.encoding_for_model("gpt-4o")

In [4]:
def count_tokens(text):
    return len(tokenizer.encode(str(text)))

df["token_count"] = df.iloc[:, -1].apply(count_tokens)
df['token_count'].describe()

count    636.000000
mean     231.816038
std      153.941350
min       52.000000
25%      105.000000
50%      185.500000
75%      332.000000
max      840.000000
Name: token_count, dtype: float64

In [5]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Report_schema(BaseModel):
    FINDINGS: str = Field(description="Contains the detailed observations organized by anatomy or template field.")
    IMPRESSION: str = Field(description="Contains the concise clinical summary: it should report the major abnormal findings and may include a possible diagnosis when that diagnosis is supported by the FINDINGS. It should not repeat every normal statement or introduce new information.")

In [30]:
# Put GEMINI_API_KEY=... in .env. Never place the key directly in this notebook.
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    raise RuntimeError('Set GEMINI_API_KEY in .env before running this notebook.')

MODEL = 'gemini-3.5-flash-lite'
INPUT_CSV = Path('test.csv')
OUTPUT_CSV = Path('final_submission_gemini.csv')
FAILURES_CSV = Path('gemini_failures.csv')

# Keep this below your Gemini API quota. Lower it if you still receive 429 errors.
REQUESTS_PER_MINUTE = 5
MAX_RETRIES = 6
INITIAL_RETRY_DELAY_SECONDS = 30
MAX_RETRY_DELAY_SECONDS = 240
MAX_OUTPUT_TOKENS = 1600

client = genai.Client(api_key=api_key)


In [31]:
SYSTEM_INSTRUCTION = '''You are a radiologist preparing a final imaging report.

Core reporting rule:-
1. Treat the template as the starting report, not merely as an example.
2. Route every dictated finding to the matching FINDINGS field.
3. Replace or modify the corresponding normal statement when the dictation describes an abnormality.
4. Preserve template statements for routinely visualized regions that were not mentioned in the dictation.
5. Update the IMPRESSION so it accurately summarizes the important abnormal findings.
6. Do not add findings unsupported by dictation or template_content. Use modality, body_part, study_description, patient_age_band, and patient_sex only as context; these columns do not supply additional findings.
7. Preserve the supplied template wording and report order whenever possible.
8. When the provided template contains an OTHER FINDINGS: field, use it for relevant findings that do not belong in another labelled field. Leave it empty when no such finding applies.

Return a JSON object with exactly two string fields: FINDINGS and IMPRESSION. as mention in schema. Do not add a preamble, explanation, commentary, markdown fence, or confidence statement.'''

GENERATION_CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    temperature=0,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    response_mime_type='application/json',
    response_schema=Report_schema,
)

def make_user_prompt(row):
    return f'''STUDY CONTEXT (context only; do not invent findings from it):
Modality: {row.modality}
Body part: {row.body_part}
Study description: {row.study_description}
Patient age bend: {row.patient_age_band}
Patient sex: {row.patient_sex}

NORMAL TEMPLATE:
{row.template_content.strip()}

DICTATION:
{row.dictation.strip()}
'''

In [32]:
last_request_started_at = 0.0
minimum_request_interval = 60 / REQUESTS_PER_MINUTE

def wait_for_request_slot():
    """Keep request starts below the configured requests-per-minute limit."""
    global last_request_started_at
    remaining = minimum_request_interval - (time.monotonic() - last_request_started_at)
    if remaining > 0:
        time.sleep(remaining)
    last_request_started_at = time.monotonic()

def retry_after_seconds(error):
    """Honor a server-provided retry delay when Gemini includes one."""
    match = re.search(r'retry[^0-9]*(\d+(?:\.\d+)?)', str(error), flags=re.IGNORECASE)
    return float(match.group(1)) if match else 0.0

def response_to_report(response):
    """Validate Gemini's JSON and convert it to the competition's text format."""
    raw_json = response.text
    payload = json.loads(raw_json)
    parsed = Report_schema.model_validate(payload)
    findings = parsed.FINDINGS.strip()
    impression = parsed.IMPRESSION.strip()
    if not findings or not impression:
        raise ValueError('Gemini returned an empty FINDINGS or IMPRESSION field.')
    return f'FINDINGS:\n{findings}\n\nIMPRESSION:\n{impression}'

def generate_one_report(row):
    """Retry transient API, quota, safety, and malformed-response failures with backoff."""
    retry_delay = INITIAL_RETRY_DELAY_SECONDS
    prompt = make_user_prompt(row)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            wait_for_request_slot()
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=GENERATION_CONFIG,
            )
            return response_to_report(response)
        except Exception as error:
            if attempt == MAX_RETRIES:
                raise RuntimeError(f'Gemini failed after {MAX_RETRIES} attempts: {error}') from error

            # Exponential backoff grows after every unsuccessful request. Jitter prevents
            # repeated retries from clustering at the same instant.
            wait_seconds = min(MAX_RETRY_DELAY_SECONDS, max(retry_delay, retry_after_seconds(error)))
            wait_seconds += random.uniform(0, min(5, wait_seconds * 0.1))
            print(f'Case {row.case_id}: attempt {attempt} failed ({error}). Retrying in {wait_seconds:.1f}s.')
            time.sleep(wait_seconds)
            retry_delay = min(MAX_RETRY_DELAY_SECONDS, retry_delay * 2)


In [33]:
test = pd.read_csv(INPUT_CSV)
print(test.columns)
required_columns = {
    'case_id', 'modality', 'body_part', 'study_description',
    'patient_age_band', 'patient_sex', 'template_content', 'dictation'
}
missing = set(test.columns) - required_columns 
if missing:
    raise ValueError(f'{INPUT_CSV} is missing columns: {sorted(missing)}')

test['report'] = pd.NA
if OUTPUT_CSV.exists():
    previous = pd.read_csv(OUTPUT_CSV, dtype={'case_id': str})
    if {'case_id', 'report'} <= set(previous.columns):
        saved_reports = previous.set_index('case_id')['report']
        test['report'] = test['case_id'].map(saved_reports)
        print(f'Resuming: {test.report.notna().sum()}/{len(test)} reports already saved.')

def save_checkpoint():
    test[['case_id', 'report']].to_csv(OUTPUT_CSV, index=False)


Index(['case_id', 'modality', 'body_part', 'study_description',
       'patient_age_band', 'patient_sex', 'template_content', 'dictation'],
      dtype='str')
Resuming: 105/132 reports already saved.


In [34]:
failures = []
pending = test.index[test["report"].isna()]
already_done = len(test) - len(pending)
started_at = time.monotonic()

print(
    f"Status: {already_done}/{len(test)} already complete. "
    f"{len(pending)} Gemini requests remaining."
)

for request_number, index in enumerate(pending, start=1):
    row = test.loc[index]
    # print(request_number)
    # print(index)
    completed_before = test["report"].notna().sum()
    remaining_before = len(test) - completed_before


    print(
        f"\nRequest {request_number}/{len(pending)} starting "
        f"| total done: {completed_before}/{len(test)} "
        f"| remaining: {remaining_before} "
        f"| case_id: {row.case_id}",
        flush=True,
    )

    try:
        test.loc[index, "report"] = generate_one_report(row)
        save_checkpoint()

        completed_now = test["report"].notna().sum()
        remaining_now = len(test) - completed_now
        elapsed_minutes = (time.monotonic() - started_at) / 60
        rate = request_number / elapsed_minutes if elapsed_minutes else 0
        eta_minutes = remaining_now / rate if rate else 0

        print(
            f"✓ Done | total: {completed_now}/{len(test)} "
            f"| left: {remaining_now} "
            f"| rate: {rate:.2f} reports/min "
            f"| ETA: ~{eta_minutes:.1f} min",
            flush=True,
        )

    except Exception as error:
        failures.append({"case_id": row.case_id, "error": str(error)})
        pd.DataFrame(failures).to_csv(FAILURES_CSV, index=False)
        print(
            f"✗ Failed | case_id: {row.case_id} "
            f"| saved error to {FAILURES_CSV}",
            flush=True,
        )

save_checkpoint()

done = test["report"].notna().sum()
left = len(test) - done
print(f"\nFinal status: {done}/{len(test)} complete; {left} left.")

if failures or left:
    raise RuntimeError(
        f"Incomplete submission: {done}/{len(test)} reports generated. "
        f"Check {FAILURES_CSV} and re-run to resume."
    )

print(f"Finished successfully: {OUTPUT_CSV}")

Status: 105/132 already complete. 27 Gemini requests remaining.

Request 1/27 starting | total done: 105/132 | remaining: 27 | case_id: 89cb3fe2-9aee-4e4a-85c3-e0e5a099b911
✓ Done | total: 106/132 | left: 26 | rate: 26.85 reports/min | ETA: ~1.0 min

Request 2/27 starting | total done: 106/132 | remaining: 26 | case_id: 8ab1a654-3318-4ec3-803a-5bf97d466f91
✓ Done | total: 107/132 | left: 25 | rate: 7.98 reports/min | ETA: ~3.1 min

Request 3/27 starting | total done: 107/132 | remaining: 25 | case_id: 8b404fa8-569c-43c9-8785-bd31382ed36a
✓ Done | total: 108/132 | left: 24 | rate: 7.03 reports/min | ETA: ~3.4 min

Request 4/27 starting | total done: 108/132 | remaining: 24 | case_id: 956291b1-faba-4c9f-991b-7cc282287ed9
✓ Done | total: 109/132 | left: 23 | rate: 5.33 reports/min | ETA: ~4.3 min

Request 5/27 starting | total done: 109/132 | remaining: 23 | case_id: 959135c4-6c14-4342-ab67-d63ffd6ed6fe
✓ Done | total: 110/132 | left: 22 | rate: 4.98 reports/min | ETA: ~4.4 min

Request 6